# Encoder-Only Transformer

**Auteur**: `Jakob De Vreese`  
**Bachelorproef**: Overdraagbare unsupervised anomaliedetectie bij hybride HVAC-systemen  
**Academiejaar**: `2025-2026`

## Doel

Implementatie van een **Encoder-Only Transformer** voor anomaliedetectie in hybride HVAC-systemen, gebaseerd op Abdollah et al. (2024). Het model wordt zelf-gesuperviseerd getraind via gemaskeerde reconstructie, zonder gelabelde foutdata.

## Stappenplan

| Stap | Inhoud |
|---|---|
| 1 | Data preprocessing (feature reductie, normalisatie, windowing) |
| 2 | Architectuur (Time2Vec, Pre-LN Transformer, reconstructie) |
| 3 | Self-supervised training (Markov-maskering, masked MSE) |
| 4 | Initiële training (baseline sanity check) |
| 5 | Hyperparameter tuning (KerasTuner Bayesiaanse optimalisatie) |
| 6 | Evaluatie van het beste model (strategievergelijking, scorekaart, eindvisualisatie) |

In [ ]:
import pandas as pd
import numpy as np
import keras
from sklearn.preprocessing import StandardScaler
import joblib
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

import random
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


## Parameters

Pas hier de configuratie aan om te schakelen naar een ander gebouw of andere instellingen. De rest van de notebook leest automatisch van deze variabelen.

In [ ]:
# ── Configureerbare parameters ─────────────────────────────────────────────
GEBOUW         = 'dunant1'  # Gebouwnaam (bestandsprefiks voor CSV en opgeslagen artefacten)
WINDOW_SIZE    = 144        # Vensterbreedte: 144 × 10 min = 24 uur
CORR_THRESHOLD = 0.97       # Correlatiedrempel voor feature-reductie
MIN_RUN        = 3          # Minimale alarmlengte voor run-length filter (tijdstappen)
NUM_EPOCHS     = 100        # Maximum epochs voor (her)training

## Stap 1 — Data Preprocessing

Voordat de data in het Keras-model wordt gevoed, wordt de volgende voorbereiding uitgevoerd:

1. **Data inlezen**
2. **Feature reductie** (correlatie > `CORR_THRESHOLD` verwijderen)
3. **Data opsplitsen** (70 / 15 / 15)
4. **Normalisatie** (Z-score, gefitst op train)
5. **Windowing** — 3D-tensoren (samples, window_size, features)

### 1.1 Dataset inlezen

In [ ]:
url = f'../02_eda_en_ground_truth/processed/{GEBOUW}_train.csv'
data = pd.read_csv(url)
data.head()

In [ ]:
print(data.shape)

### 1.2 Feature reductie

Een feature per paar met correlatie **> `CORR_THRESHOLD`** wordt verwijderd — zelfde drempel als TranAD. Dit reduceert redundantie en verlaagt de modelcomplexiteit zonder informatieverlies.

In [ ]:
import json

# Correlatie berekend uitsluitend op de trainset (eerste 70%) — geen lekkage van val/test
n_train = int(len(data) * 0.70)
train_for_corr = data.iloc[:n_train].drop(columns=["timestamp"])

corr_matrix = train_for_corr.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > CORR_THRESHOLD)]
kept_features = [col for col in data.columns if col not in to_drop and col != "timestamp"]

print(f"Behouden features: {len(kept_features)} (drempel: >{CORR_THRESHOLD})")

import os
os.makedirs('encoder_only', exist_ok=True)
with open(f"encoder_only/features_{GEBOUW}.json", "w") as f:
    json.dump(kept_features, f)

data_filtered = data[kept_features]

### 1.3 Data opspliten

In [ ]:
n = len(data)
train_df = data_filtered[0:int(n*0.7)]
val_df = data_filtered[int(n*0.7):int(n*0.85)]
test_df = data_filtered[int(n*0.85):]

### 1.4 Normalisatie (Z-score Scaling)

Volgens de paper moeten we de data standaardiseren. We normaliseren `train_df` en gebruiken de parameters om dit ook op de rest toe te passen.

In [ ]:
scaler = StandardScaler()

# Fit alleen op training data
scaler.fit(train_df)

# Transformeer alle sets
train_scaled = scaler.transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

# Sla de scaler op voor later gebruik in productie!
joblib.dump(scaler, f'encoder_only/scaler_{GEBOUW}.pkl')

### 1.5 Windowing (3D Tensor maken)

De Transformer heeft een venster nodig van data om patronen te herkennen. In de paper gebruikt men venstergoottes ($w$) van 1 dag of wel **`WINDOW_SIZE`** tijdstappen (standaard 144 bij 10-minuten intervallen).

In [ ]:
def create_windows(data_array, window_size=144):
    X = []
    for i in range(len(data_array) - window_size):
        X.append(data_array[i:i + window_size])
    return np.array(X)

X_train = create_windows(train_scaled, window_size=WINDOW_SIZE)
X_val   = create_windows(val_scaled,   window_size=WINDOW_SIZE)
X_test  = create_windows(test_scaled,  window_size=WINDOW_SIZE)

print(f"Shape training data: {X_train.shape}")

## Stap 2 — Architectuur

De architectuur implementeert een **encoder-only Transformer** voor unsupervised anomaliedetectie, gebaseerd op Figuur 1 en Tabel 1 van de paper. Ten opzichte van de paper zijn twee verbeteringen doorgevoerd:

1. **Pre-LayerNorm** in plaats van Post-LN — stabielere training op kleine datasets, kleinere train/val-kloof
2. **Finale LayerNorm** na de encoder-stack — standaardpraktijk bij Pre-LN Transformers

### Data-doorstroom
`X ∈ ℝ^(w×m)` → Norm → Projectie → Time2Vec → `B × Encoder` → Finale LN → Reconstructie → `X̂ ∈ ℝ^(w×m)`

### Time2Vec — Tijdscodering

**Time2Vec** codeert de positie van elk tijdstap als een leerbare vector [(Kazemi et al., 2019)](https://arxiv.org/abs/1907.05321). Het combineert één lineaire component (trend) met $k$ periodieke sinusoïden:

$$\text{t2v}(\tau)[i] = \begin{cases} \omega_i \tau + \varphi_i & i = 0 \\ \sin(\omega_i \tau + \varphi_i) & 1 \leq i \leq k \end{cases}$$

De uitvoer $W_{\text{pos}} \in \mathbb{R}^{w \times d}$ wordt opgeteld bij de lineaire projectie $U$, zodat het model tijdsstructuur kan meenemen. In de paper: `kernel_size = d_model − 1`.

In [ ]:
class Time2Vec(layers.Layer):
    def __init__(self, output_dim):
        super(Time2Vec, self).__init__()
        self.output_dim = output_dim  # = d_model

    def build(self, input_shape):
        # Lineaire component (trend)
        self.wb = self.add_weight(name='wb', shape=(1,), initializer='uniform', trainable=True)
        self.bb = self.add_weight(name='bb', shape=(1,), initializer='uniform', trainable=True)
        # Periodieke componenten: k = output_dim - 1 sinusoïden
        self.wa = self.add_weight(name='wa', shape=(1, self.output_dim - 1), initializer='uniform', trainable=True)
        self.ba = self.add_weight(name='ba', shape=(1, self.output_dim - 1), initializer='uniform', trainable=True)

    def call(self, t):
        # t: (window_size, 1) — tijdstapindices [0, 1, ..., w-1]
        v1 = self.wb * t + self.bb          # lineair:   (window_size, 1)
        v2 = tf.sin(self.wa * t + self.ba)  # periodiek: (window_size, output_dim-1)
        return tf.concat([v1, v2], axis=-1) # (window_size, output_dim)


### Transformer Encoder Blok (Pre-LayerNorm)

We gebruiken **Pre-LN** in plaats van de Post-LN uit de paper: de LayerNorm staat vóór het sublayer, de residualverbinding erna. Schema per deellaag:

$$\text{output} = \text{sublayer}(\text{LN}(x)) + x$$

Bij Post-LN worden gradiënten geblokkeerd door de norm ná de residual, wat leidt tot instabiliteit op kleine datasets. Pre-LN laat gradiënten ongehinderd door de skip-verbindingen stromen en verkleint de train/val-kloof aanzienlijk [(Ba et al., 2016)](https://arxiv.org/abs/1607.06450).

Elk blok bevat:
1. Pre-LN + **Multi-Head Self-Attention** — leert temporele afhankelijkheden
2. Pre-LN + **Point-wise FFN** — niet-lineaire feature-transformatie per tijdstap

In [ ]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Pre-LayerNorm: LN vóór het sublayer, residual erna
    # Stabieler dan Post-LN op kleine datasets; verkleint de train/val gap

    # 1. Pre-LN + Multi-Head Attention
    residual = inputs
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Dropout(dropout)(x)
    x = x + residual

    # 2. Pre-LN + Point-wise FFN
    residual = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(ff_dim, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(inputs.shape[-1])(x)
    x = x + residual

    return x


### Het volledige model

De volledige architectuur volgt Figuur 1 van de paper met Pre-LN correctie:

- **`key_dim = d_model // num_heads`** — standaard Transformer: de aandacht wordt verdeeld over alle heads
- **Finale `LayerNormalization`** na de encoder-stack, vóór de reconstructieprojectie

In [ ]:
def build_model(window_size, num_features, d_model, num_heads, ff_dim, num_layers, dropout):
    inputs = layers.Input(shape=(window_size, num_features))

    # --- STAP 1: Preprocessing & Encoding ---
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.Dense(d_model)(x)  # lineaire projectie → U

    # Time2Vec positional encoding: Z = U + W_pos
    positions = tf.cast(tf.reshape(tf.range(window_size), (window_size, 1)), tf.float32)
    pos_encoding = Time2Vec(output_dim=d_model)(positions)
    x = x + pos_encoding

    # --- STAP 2: Transformer Processing (Pre-LN blokken) ---
    for _ in range(num_layers):
        x = transformer_encoder(x, d_model // num_heads, num_heads, ff_dim, dropout)

    # Finale LayerNorm na de encoder stack (standaard bij Pre-LN Transformers)
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    # --- STAP 3: Reconstruction Step ---
    outputs = layers.Dense(num_features)(x)

    return tf.keras.Model(inputs, outputs)


In [ ]:
# Architectuuroverzicht met voorbeeldparameters
model = build_model(
    window_size=WINDOW_SIZE, num_features=len(kept_features),
    d_model=64, num_heads=8, ff_dim=128, num_layers=2, dropout=0.1
)
model.summary()

## Stap 3 — Self-Supervised Pre-training

### Markov Chain Masking

Het model wordt getraind als **masked autoencoder**: een deel van de tijdreeks wordt verborgen en het model leert de verborgen segmenten te reconstrueren vanuit de zichtbare context (§3.2). De maskeermatrix $M \in \{0,1\}^{w \times m}$ wordt per feature gegenereerd via een Markov-keten:

$$l_u = \frac{1-r}{r} \cdot l_m, \qquad p_m = \frac{1}{l_u}, \qquad p_u = \frac{1}{l_m}$$

- $l_m$: gemiddelde lengte gemaskeerd segment (paper: $l_m = 3$)
- $r$: proportie te maskeren data (paper: $r = 0.15$)
- Gemaskeerde posities worden vervangen door **nul** (= trainingsgemiddelde na Z-score normalisatie)

In [ ]:
def generate_markov_mask(input_shape, r=0.15, lm=3):
    """
    Genereert een binaire mask matrix (M) op basis van een Markov Chain.
    Input_shape: (window_size, num_features)
    """
    W, M = input_shape
    # Bereken de gemiddelde ongemaskeerde lengte
    lu = ((1 - r) / r) * lm
    
    # Transitie kansen (p_m: 1->0, p_u: 0->1)
    p_m = 1 / lu
    p_u = 1 / lm
    
    mask = np.ones(input_shape)
    
    for j in range(M): # Per feature
        curr_state = 1 # Start als unmasked
        for i in range(W):
            if curr_state == 1:
                if np.random.rand() < p_m:
                    curr_state = 0
            else:
                if np.random.rand() < p_u:
                    curr_state = 1
            mask[i, j] = curr_state
            
    return mask

### Masked MSE Loss

De verliesfunctie wordt **uitsluitend berekend op de gemaskeerde posities** ($M_{ti} = 0$), conform vergelijking 8 van de paper:

$$\mathcal{L} = \frac{1}{|\mathcal{M}|} \sum_{(t,i) \in \mathcal{M}} \bigl(\hat{x}(t,i) - x(t,i)\bigr)^2$$

Dit dwingt het model om verborgen waarden te voorspellen vanuit context — niet om zichtbare waarden te kopiëren.

In [ ]:
def masked_mse_loss(y_true, y_pred, mask):
    """
    Berekent MSE uitsluitend op de gemaskeerde segmenten.
    """
    # De paper berekent loss enkel op de verborgen delen (waar mask == 0)
    # We maken een invers masker: 1 voor verborgen, 0 voor zichtbaar
    inverse_mask = 1.0 - tf.cast(mask, tf.float32)
    
    sq_diff = tf.square(y_true - y_pred) * inverse_mask
    
    # Gemiddelde nemen over het aantal gemaskeerde punten
    loss = tf.reduce_sum(sq_diff) / (tf.reduce_sum(inverse_mask) + 1e-6)
    return loss

### HVACModel — Trainings-wrapper

`HVACModel` inkapselt de Transformer en overschrijft `train_step` en `test_step` om Markov-maskering toe te passen.

**Kritiek implementatiedetail**: het masker wordt **per sample onafhankelijk** gegenereerd via `generate_batch_masks`. Bij één gedeeld masker per batch zijn alle gradiëntsignalen gecorreleerd — het model leert dan vaste posities te negeren in plaats van de onderliggende tijdreeksdistributie.

In [ ]:
def generate_batch_masks(bs, ws, nf, r, lm):
    """Genereert een uniek Markov-masker per sample in de batch."""
    bs, ws, nf = int(bs), int(ws), int(nf)
    r, lm = float(r), float(lm)
    return np.array(
        [generate_markov_mask([ws, nf], r, lm) for _ in range(bs)],
        dtype=np.float64
    )


class HVACModel(tf.keras.Model):
    def __init__(self, transformer_model, r=0.15, lm=3):
        super(HVACModel, self).__init__()
        self.transformer = transformer_model
        self.r = r
        self.lm = lm
        self.loss_tracker = tf.keras.metrics.Mean(name='masked_mse')

    def _make_mask_batch(self, batch_size, window_size, num_features):
        """Roept generate_batch_masks aan via tf.numpy_function."""
        mask_batch = tf.numpy_function(
            func=generate_batch_masks,
            inp=[batch_size, window_size, num_features, self.r, self.lm],
            Tout=tf.float64
        )
        mask_batch = tf.cast(mask_batch, dtype=tf.float32)
        mask_batch.set_shape([None, window_size, num_features])
        return mask_batch

    def train_step(self, data):
        x = data
        window_size = x.shape[1]
        num_features = x.shape[2]
        batch_size = tf.shape(x)[0]

        # Uniek masker per sample (fix: was voorheen één masker voor hele batch)
        mask_batch = self._make_mask_batch(batch_size, window_size, num_features)

        x_masked = x * mask_batch
        with tf.GradientTape() as tape:
            y_pred = self.transformer(x_masked, training=True)
            loss = masked_mse_loss(x, y_pred, mask_batch)

        gradients = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {'masked_mse': self.loss_tracker.result()}

    def test_step(self, data):
        x, y = data
        window_size = x.shape[1]
        num_features = x.shape[2]
        batch_size = tf.shape(x)[0]

        mask_batch = self._make_mask_batch(batch_size, window_size, num_features)

        x_masked = x * mask_batch
        y_pred = self.transformer(x_masked, training=False)
        loss = masked_mse_loss(x, y_pred, mask_batch)
        self.loss_tracker.update_state(loss)
        return {'masked_mse': self.loss_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker]

    def call(self, inputs):
        return self.transformer(inputs)


In [ ]:
# Baseline model (paper-parameters)
base_transformer = build_model(
    window_size=WINDOW_SIZE, num_features=len(kept_features),
    d_model=64, num_heads=8, ff_dim=128, num_layers=2, dropout=0.1
)
hvac_model = HVACModel(base_transformer, r=0.15, lm=3)
hvac_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))
hvac_model.build(input_shape=(None, WINDOW_SIZE, len(kept_features)))
hvac_model.summary()

## Stap 4 — Initiële Training (Baseline)

We trainen eerst een model met vaste paper-parameters als **sanity check**: werkt de architectuur en trainingsloop correct? Dit model wordt daarna vervangen door het beste model uit de hyperparameter-tuning (Stap 5).

### 4.1 Callbacks

- **`EarlyStopping`** (patience=10): stopt zodra `val_masked_mse` niet meer daalt; herstelt de beste gewichten
- **`ModelCheckpoint`**: slaat de beste gewichten op bij elke verbetering op de validatieset
- **`ReduceLROnPlateau`** (patience=5, factor=0.5): halveert de leersnelheid bij stagnatie

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# 1. Optimizer: Adam met een bescheiden leersnelheid (LR) zoals in de paper
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# 2. EarlyStopping: Stop als de validatie-loss 10 epochs niet meer zakt
early_stopping = EarlyStopping(
    monitor='val_masked_mse', 
    patience=10, 
    restore_best_weights=True,
    mode='min'
)

# 3. ModelCheckpoint: Sla de beste gewichten op in een bestand
checkpoint = ModelCheckpoint(
    filepath=f'encoder_only/best_model_{GEBOUW}.weights.h5',
    monitor='val_masked_mse',
    save_best_only=True,
    save_weights_only=True, # We slaan gewichten op omdat we een custom model gebruiken
    mode='min'
)

# 4. ReduceLROnPlateau: Verlaag de LR als de training stagneert
reduce_lr = ReduceLROnPlateau(
    monitor='val_masked_mse', 
    factor=0.5, 
    patience=5, 
    min_lr=1e-6,
    mode='min'
)

callbacks = [early_stopping, checkpoint, reduce_lr]

### 4.2 Training

`X_train` dient zowel als input als reconstructiedoel. De `train_step` genereert per batch een nieuw masker per sample en berekent de masked MSE op de verborgen posities.

In [ ]:
history = hvac_model.fit(
    X_train,
    epochs=NUM_EPOCHS,
    batch_size=32,
    validation_data=(X_val, X_val),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
def plot_history(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['masked_mse'], label='Train Masked MSE')
    plt.plot(history.history['val_masked_mse'], label='Validation Masked MSE')
    plt.title('Model Training Progress (Self-Supervised Reconstructie)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(history)

### 4.3 Scorekaart initieel model

We evalueren het initiële baseline model op de synthetische testset als **sanity check** vóór hyperparameter-tuning. De drempel wordt gekalibreerd via F2-maximalisatie op de tune-split (eerste helft van de testset) — identiek aan de aanpak in TranAD stap 4.

In [ ]:
from scipy.stats import genpareto
from sklearn.metrics import (precision_recall_fscore_support, balanced_accuracy_score,
                              matthews_corrcoef, roc_auc_score, average_precision_score,
                              fbeta_score as _fb_bl, confusion_matrix as _cm_bl)

# ── Helper functies (lokaal benoemd om naamconflicten met stap 6 te vermijden) ──
def _get_event_stats_bl(y_true, y_pred):
    def _segs(y):
        segs, in_seg = [], False
        for i, v in enumerate(y):
            if v and not in_seg: in_seg, start = True, i
            elif not v and in_seg: segs.append((start, i - 1)); in_seg = False
        if in_seg: segs.append((start, len(y) - 1))
        return segs
    true_ev = _segs(y_true); pred_ev = _segs(y_pred)
    detected = sum(any(not (pe < ts or ps > te) for ps, pe in pred_ev) for ts, te in true_ev)
    correct  = sum(any(not (te < ps or ts > pe) for ts, te in true_ev) for ps, pe in pred_ev)
    er = detected / len(true_ev) if true_ev else 0.0
    ep = correct  / len(pred_ev) if pred_ev else 0.0
    ef = 2 * ep * er / (ep + er) if (ep + er) > 0 else 0.0
    return {"total_events": len(true_ev), "detected_events": detected,
            "n_pred_events": len(pred_ev), "event_recall": er, "event_precision": ep, "event_f1": ef}

def _rl_filter_bl(y_pred, min_run=MIN_RUN):
    filtered = y_pred.copy().astype(int); i = 0
    while i < len(filtered):
        if filtered[i] == 1:
            rs = i
            while i < len(filtered) and filtered[i] == 1: i += 1
            if (i - rs) < min_run: filtered[rs:i] = 0
        else: i += 1
    return filtered

def _wins2ts_bl(win_scores, n_timesteps, ws=WINDOW_SIZE):
    out = np.zeros(n_timesteps); cnt = np.zeros(n_timesteps)
    for i, s in enumerate(win_scores):
        out[i:i + ws] += s; cnt[i:i + ws] += 1
    return out / np.maximum(cnt, 1)

# ── Testdata inladen ──────────────────────────────────────────────────────────
synth_df_bl     = pd.read_csv(f'../02_eda_en_ground_truth/processed/{GEBOUW}_test.csv')
y_true_ts_bl    = np.load(f'../02_eda_en_ground_truth/processed/{GEBOUW}_test_labels.npy').astype(int)
synth_scaled_bl = scaler.transform(synth_df_bl[kept_features])
X_eval_bl       = create_windows(synth_scaled_bl, window_size=WINDOW_SIZE)
y_true_win_bl   = np.array([
    1 if np.any(y_true_ts_bl[i:i + WINDOW_SIZE] == 1) else 0
    for i in range(len(y_true_ts_bl) - WINDOW_SIZE)
])
n_ts_bl = len(y_true_ts_bl)
print(f"Eval windows: {X_eval_bl.shape}  |  anomalie-ratio: {y_true_ts_bl.mean():.3f}")

# ── Val-statistieken voor z-score normalisatie ────────────────────────────────
val_rec_bl      = hvac_model.transformer.predict(X_val, verbose=0)
val_feat_mse_bl = np.mean(np.square(X_val - val_rec_bl), axis=1)
feat_mean_bl    = val_feat_mse_bl.mean(axis=0)
feat_std_bl     = val_feat_mse_bl.std(axis=0) + 1e-8

# ── Inferentie + scoring strategieën ─────────────────────────────────────────
rec_bl = hvac_model.transformer.predict(X_eval_bl, verbose=0)
sq_bl  = np.square(X_eval_bl - rec_bl)
fm_bl  = np.mean(sq_bl, axis=1)   # (n_wins, n_feat)
ts_bl  = np.mean(sq_bl, axis=2)   # (n_wins, 144)
fn_bl  = (fm_bl - feat_mean_bl) / feat_std_bl

strategies_bl = {
    "max-feat (raw)": np.max(fm_bl, axis=1),
    "global-mean":    np.mean(sq_bl, axis=(1, 2)),
    "max-ts":         np.max(ts_bl, axis=1),
    "p95-feat":       np.percentile(fm_bl, 95, axis=1),
    "norm-mean":      fn_bl.mean(axis=1),
    "norm-max":       fn_bl.max(axis=1),
}

print(f"\n{'Strategie':22s}  {'ROC-AUC (win)':14s}  {'ROC-AUC (ts)':12s}  Nota")
best_roc_bl, best_name_bl, best_win_bl = 0.0, '', None
for nm, sc in strategies_bl.items():
    ts_sc = _wins2ts_bl(sc, n_ts_bl)
    roc_win = roc_auc_score(y_true_win_bl, sc)
    roc_ts  = roc_auc_score(y_true_ts_bl, ts_sc)
    eff = max(roc_ts, 1 - roc_ts); flip = roc_ts < 0.5
    print(f"  {nm:22s}  {roc_win:.4f}          {roc_ts:.4f}  {'↑FLIP' if flip else ''}")
    use_sc = -sc if flip else sc
    if eff > best_roc_bl: best_roc_bl, best_name_bl, best_win_bl = eff, nm, use_sc
print(f"\nBeste strategie: '{best_name_bl}'  (effectieve ROC-AUC ts = {best_roc_bl:.4f})")

# ── Smoothing + tune/eval split ───────────────────────────────────────────────
_k = np.array([1, 2, 3, 2, 1], dtype=float); _k /= _k.sum()
best_ts_bl   = np.convolve(_wins2ts_bl(best_win_bl, n_ts_bl), _k, mode='same')
nm_max_ts_bl = np.convolve(_wins2ts_bl(strategies_bl['norm-max'], n_ts_bl), _k, mode='same')

anom_pos_bl  = np.where(y_true_ts_bl == 1)[0]
split_bl     = int(np.median(anom_pos_bl)) if len(anom_pos_bl) >= 4 else n_ts_bl // 2
y_tune_bl    = y_true_ts_bl[:split_bl];    y_eval_bl    = y_true_ts_bl[split_bl:]
sc_tune_bl   = best_ts_bl[:split_bl];      sc_eval_bl   = best_ts_bl[split_bl:]
sc_tune_nm   = nm_max_ts_bl[:split_bl];    sc_eval_nm   = nm_max_ts_bl[split_bl:]
print(f"\nSplit: {split_bl}  |  anomalieën tune: {y_tune_bl.sum()}  eval: {y_eval_bl.sum()}")

# ── F2-maximalisatie op tune-set (methode A & B) ──────────────────────────────
cands_a = np.percentile(sc_tune_bl, np.linspace(0.1, 99.9, 600))
best_f2_a_bl, thr_a_bl = 0.0, cands_a[-1]
for t in cands_a:
    f2v = _fb_bl(y_tune_bl, (sc_tune_bl > t).astype(int), beta=2, zero_division=0)
    if f2v > best_f2_a_bl: best_f2_a_bl, thr_a_bl = f2v, t

cands_b = np.percentile(sc_tune_nm, np.linspace(0.1, 99.9, 600))
best_f2_b_bl, thr_b_bl = 0.0, cands_b[-1]
for t in cands_b:
    yp  = _rl_filter_bl((sc_tune_nm > t).astype(int))
    f2v = _fb_bl(y_tune_bl, yp, beta=2, zero_division=0)
    if f2v > best_f2_b_bl: best_f2_b_bl, thr_b_bl = f2v, t

print(f"\nMethode A — {best_name_bl:<25s}  tune F2 = {best_f2_a_bl:.4f}")
print(f"Methode B — norm-max + RL({MIN_RUN})               tune F2 = {best_f2_b_bl:.4f}")

if best_f2_b_bl >= best_f2_a_bl:
    print("→ Methode B gekozen")
    final_sc_bl      = sc_eval_nm
    final_thr_bl     = thr_b_bl
    chosen_method_bl = f"norm-max + RL({MIN_RUN})"
    y_pred_eval_bl   = _rl_filter_bl((sc_eval_nm > thr_b_bl).astype(int))
else:
    print(f"→ Methode A gekozen ({best_name_bl})")
    final_sc_bl      = sc_eval_bl
    final_thr_bl     = thr_a_bl
    chosen_method_bl = best_name_bl
    y_pred_eval_bl   = (sc_eval_bl > thr_a_bl).astype(int)

# ── Scorekaart samenstellen ───────────────────────────────────────────────────
has_both_bl = len(np.unique(y_eval_bl)) > 1
p_bl, r_bl, f1_bl, _ = precision_recall_fscore_support(y_eval_bl, y_pred_eval_bl, average='binary', zero_division=0)
f2_bl  = _fb_bl(y_eval_bl, y_pred_eval_bl, beta=2, zero_division=0)
ba_bl  = balanced_accuracy_score(y_eval_bl, y_pred_eval_bl)
mcc_bl = matthews_corrcoef(y_eval_bl, y_pred_eval_bl)
roc_bl = roc_auc_score(y_eval_bl, final_sc_bl)           if has_both_bl else float('nan')
prc_bl = average_precision_score(y_eval_bl, final_sc_bl) if has_both_bl else float('nan')
evt_bl = _get_event_stats_bl(y_eval_bl, y_pred_eval_bl)

bl_result = {
    'metrics_df': pd.DataFrame([
        {'Metric': 'Precision',             'Value': p_bl},
        {'Metric': 'Recall',                'Value': r_bl},
        {'Metric': 'F1-Score',              'Value': f1_bl},
        {'Metric': 'F2-Score (HVAC Focus)', 'Value': f2_bl},
        {'Metric': 'Balanced Accuracy',     'Value': ba_bl},
        {'Metric': 'MCC',                   'Value': mcc_bl},
        {'Metric': 'ROC-AUC',               'Value': roc_bl},
        {'Metric': 'PR-AUC',                'Value': prc_bl},
        {'Metric': 'Event Recall',          'Value': evt_bl['event_recall']},
        {'Metric': 'Event Precision',       'Value': evt_bl['event_precision']},
        {'Metric': 'Event F1',              'Value': evt_bl['event_f1']},
    ]),
    'confusion_matrix': _cm_bl(y_eval_bl, y_pred_eval_bl),
    'y_pred': y_pred_eval_bl,
    'event_details': evt_bl,
    'threshold': float(final_thr_bl),
}
print(f"\nEncoder-Only Baseline — methode: {chosen_method_bl}  |  drempel: {final_thr_bl:.4f}")
print(bl_result['metrics_df'].to_string(index=False))
print(f"\nAlarm rate eval: {y_pred_eval_bl.mean():.3f}  vs  GT: {y_eval_bl.mean():.3f}")
print(f"Event details: {evt_bl}")

In [ ]:
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
fig.suptitle(f'Encoder-Only Baseline — {GEBOUW} — Scorekaart', fontsize=18, fontweight='bold')

ax = axes[0, 0]
ax.plot(final_sc_bl, lw=1.0, color='#1f77b4', label=chosen_method_bl)
ax.axhline(final_thr_bl, color='#d62728', ls='--', lw=2,
           label=f'Drempel = {final_thr_bl:.4f}')
anom = np.where(y_eval_bl == 1)[0]
ax.scatter(anom, final_sc_bl[anom], s=12, color='#ff7f0e', alpha=0.7, label='GT anomalie', zorder=5)
ax.set_title('Anomaliescores per tijdstap')
ax.set_xlabel('Tijdstap')
ax.set_ylabel(chosen_method_bl)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)

ax = axes[0, 1]
for lbl, clr, nm in [(0, '#1f77b4', 'Normaal'), (1, '#d62728', 'Anomalie')]:
    mask = y_eval_bl == lbl
    if mask.any():
        sns.histplot(final_sc_bl[mask], bins=40, kde=True, stat='density',
                     color=clr, alpha=0.45, label=nm, ax=ax)
ax.axvline(final_thr_bl, color='#2ca02c', ls='--', lw=2, label='Drempel')
ax.set_title('Scoreverdeling per klasse')
ax.set_xlabel('Score')
ax.legend(fontsize=10)
ax.grid(alpha=0.25)

ax = axes[1, 0]
sns.heatmap(bl_result['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            cbar=False, square=True,
            xticklabels=['Pred normaal', 'Pred anomalie'],
            yticklabels=['Echt normaal', 'Echt anomalie'], ax=ax)
ax.set_title('Confusion matrix')

ax = axes[1, 1]
mdf = bl_result['metrics_df'].sort_values('Value', ascending=True)
sns.barplot(data=mdf, x='Value', y='Metric', hue='Metric',
            dodge=False, palette='viridis', legend=False, ax=ax)
for i, v in enumerate(mdf['Value']):
    if not np.isnan(v):
        ax.text(min(v + 0.01, 1.05), i, f'{v:.3f}', va='center', fontsize=10)
ax.set_xlim(0, 1.15)
ax.set_title('Kernmetrieken')
ax.set_xlabel('Score')
plt.show()

## Stap 5 — Hyperparameter Tuning

De architectuur en het trainingsproces bevatten meerdere parameters die de detectieprestaties sterk beïnvloeden. We gebruiken **KerasTuner met Bayesiaanse Optimalisatie**: elk nieuw trial-punt wordt gekozen op basis van het Gaussiaans-proces over de al gevonden resultaten, wat efficiënter convergeert dan willekeurig zoeken.

### 5.1 Zoekruimte

| Parameter | Opties | Beschrijving |
|---|---|---|
| `d_model` | 64, 128 | Interne modeldimensie |
| `num_heads` | 4, 8 | Aantal attention-heads (`key_dim = d_model / num_heads`) |
| `ff_dim` | 64, 128, 256 | Breedte van het point-wise FFN |
| `num_layers` | 1 – 3 | Aantal gestapelde encoder-blokken |
| `dropout` | 0.0, 0.1, 0.2 | Dropout-regularisatie |
| `learning_rate` | 1e-4, 3e-4, 1e-3 | Initiële Adam-leersnelheid |
| `r` | 0.10, 0.15, 0.20 | Maskerverhouding (proportie verborgen data) |
| `lm` | 2, 3, 5 | Gemiddelde lengte van een gemaskeerd segment |

In [ ]:
import keras_tuner as kt

def build_hvac_hypermodel(hp):
    d_model    = hp.Choice('d_model', [64, 128])
    num_heads  = hp.Choice('num_heads', [4, 8])
    ff_dim     = hp.Choice('ff_dim', [64, 128, 256])
    num_layers = hp.Int('num_layers', min_value=1, max_value=3)
    dropout    = hp.Float('dropout', min_value=0.0, max_value=0.2, step=0.1)
    lr         = hp.Choice('learning_rate', [1e-4, 3e-4, 1e-3])
    r          = hp.Choice('r', [0.10, 0.15, 0.20])
    lm         = hp.Choice('lm', [2, 3, 5])

    base = build_model(
        window_size=WINDOW_SIZE, num_features=len(kept_features),
        d_model=d_model, num_heads=num_heads, ff_dim=ff_dim,
        num_layers=num_layers, dropout=dropout
    )
    model = HVACModel(base, r=r, lm=lm)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr))
    model.build(input_shape=(None, WINDOW_SIZE, len(kept_features)))
    return model

### 5.2 Zoekproces

Elke trial traint maximaal 40 epochs met `EarlyStopping` (patience=8). De tuner minimaliseert `val_masked_mse` over 30 trials. Dit duurt ca. **15-45 minuten** op CPU afhankelijk van de hardware.

In [ ]:
tuner = kt.BayesianOptimization(
    hypermodel=build_hvac_hypermodel,
    objective=kt.Objective('val_masked_mse', direction='min'),
    max_trials=30,
    directory='tuning_logs',
    project_name='encoder_only_tuning',
    overwrite=True
)

tuner.search(
    X_train,
    epochs=40,
    batch_size=32,
    validation_data=(X_val, X_val),
    callbacks=[
        EarlyStopping(monitor='val_masked_mse', patience=8,
                      restore_best_weights=True, mode='min', verbose=0)
    ],
    verbose=0
)
tuner.results_summary(num_trials=5)


### 5.3 Beste configuratie volledig hertrain

De tuner selecteert de beste hyperparameters op basis van de kortste trials (max 40 epochs, EarlyStopping patience=8). We hertrain het beste model volledig met de volledige callbacks uit Stap 4 en slaan het op in dezelfde conventie als TranAD.

In [ ]:
import os
from datetime import datetime

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print('Beste hyperparameters:')
for key, value in best_hp.values.items():
    print(f'  {key}: {value}')

best_base = build_model(
    window_size=WINDOW_SIZE, num_features=len(kept_features),
    d_model=best_hp.get('d_model'),
    num_heads=best_hp.get('num_heads'),
    ff_dim=best_hp.get('ff_dim'),
    num_layers=best_hp.get('num_layers'),
    dropout=best_hp.get('dropout')
)
best_model = HVACModel(best_base,
                        r=best_hp.get('r'),
                        lm=best_hp.get('lm'))

os.makedirs('models', exist_ok=True)
today = datetime.now().strftime('%Y-%m-%d')
save_path = f'models/encoder-only-{GEBOUW}-{today}-best.weights.h5'

best_checkpoint = ModelCheckpoint(
    filepath=save_path,
    monitor='val_masked_mse', save_best_only=True,
    save_weights_only=True, mode='min'
)
best_callbacks = [
    EarlyStopping(monitor='val_masked_mse', patience=10,
                  restore_best_weights=True, mode='min'),
    best_checkpoint,
    ReduceLROnPlateau(monitor='val_masked_mse', factor=0.5,
                      patience=5, min_lr=1e-6, mode='min'),
]

best_model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=best_hp.get('learning_rate')))
best_model.build(input_shape=(None, WINDOW_SIZE, len(kept_features)))

history_best = best_model.fit(
    X_train, epochs=NUM_EPOCHS, batch_size=32,
    validation_data=(X_val, X_val),
    callbacks=best_callbacks, verbose=1
)
plot_history(history_best)

print(f'\nBeste model opgeslagen: {save_path}')

## Stap 6 — Evaluatie van het Beste Model

We evalueren het hertrainde beste model op de synthetische gelabelde testset. De drempelwaarde wordt gekalibreerd op de **validatieset** (volledig normaal, nooit gezien door het model) en daarna toegepast op de evaluatieset.

### 6.1 Ground truth en evaluatiedata laden

In [ ]:
 # Gelabelde synthetische evaluatieset inladen
synth_csv = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test.csv'
labels_npy = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test_labels.npy'

synth_df = pd.read_csv(synth_csv)
y_true_timestep = np.load(labels_npy).astype(int) 

In [ ]:
import json
import joblib

with open(f'encoder_only/features_{GEBOUW}.json', 'r') as f:
    kept_features = json.load(f)

scaler = joblib.load(f'encoder_only/scaler_{GEBOUW}.pkl')

synth_df_filtered = synth_df[kept_features]
synth_scaled = scaler.transform(synth_df_filtered)

X_eval = create_windows(synth_scaled, window_size=WINDOW_SIZE)

print(f"Shape van de evaluatie windows: {X_eval.shape}")

In [ ]:
def align_labels_to_windows(y_true_timestep, window_size=144):
    """Converteert tijdstap-labels naar window-labels."""
    y_true_window = []
    for i in range(len(y_true_timestep) - window_size):
        window_label = 1 if np.any(y_true_timestep[i : i + window_size] == 1) else 0
        y_true_window.append(window_label)
    return np.array(y_true_window)

y_true_window = align_labels_to_windows(y_true_timestep, window_size=WINDOW_SIZE)
print(f"Shape van de window labels: {y_true_window.shape}")

### 6.2 Anomaliescores berekenen

Bij inferentie wordt het model gerund **zonder maskering** — de volledige input is zichtbaar en de reconstructiefout meet hoe goed het model normaal gedrag modelleer.

We gebruiken de **per-feature max MSE** in plaats van de globale mean:
1. Bereken voor elke window de MSE per feature (gemiddeld over 144 tijdstappen)
2. Neem de maximale MSE over alle features

Dit behoudt het signaal van gelokaliseerde fouten (bijv. bevroren sensor, vast ventiel) die bij een globale mean over 44 features worden verdund.

In [ ]:
reconstructions = best_model.transformer.predict(X_eval, verbose=0)
sq_err = np.square(X_eval - reconstructions)  # (n_wins, 144, 44)

# Per-feature time-averaged MSE and per-timestep feature-averaged MSE
feat_mse = np.mean(sq_err, axis=1)   # (n_wins, 44): average over 144 timesteps
ts_mse   = np.mean(sq_err, axis=2)   # (n_wins, 144): average over 44 features

# Four candidate scoring strategies
eval_scores_max_feat = np.max(feat_mse, axis=1)          # max over features
eval_scores_global   = np.mean(sq_err, axis=(1, 2))      # global mean MSE
eval_scores_max_ts   = np.max(ts_mse, axis=1)            # max over timesteps (catches fault transitions)
eval_scores_p95_feat = np.percentile(feat_mse, 95, axis=1)  # 95th pct over features

print(f"Windows: {len(eval_scores_global)}")
for nm, sc in [('max-feat (raw)', eval_scores_max_feat),
               ('global-mean',   eval_scores_global),
               ('max-ts',        eval_scores_max_ts),
               ('p95-feat',      eval_scores_p95_feat)]:
    print(f"  {nm:20s}: mean={sc.mean():.4f}  95p={np.percentile(sc, 95):.4f}")

### 6.3 Drempelkalibratie via POT

De anomaliedrempel wordt bepaald via de **Peak Over Threshold (POT)** methode, conform §3.3. De drempel wordt gefitst op de **validatieset** (volledig normaal), niet op de evaluatieset — anders bevat de drempelcalibratie anomalies en is de evaluatie circulair.

In [ ]:
from scipy.stats import genpareto
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

# ── Val reconstruction stats for z-score normalisation ──────────────────────
val_reconstructions = best_model.transformer.predict(X_val, verbose=0)
val_sq_err   = np.square(X_val - val_reconstructions)
val_feat_mse = np.mean(val_sq_err, axis=1)   # (n_val, n_feat)
val_ts_mse   = np.mean(val_sq_err, axis=2)   # (n_val, 144) — per-position stats

feat_mean_v = val_feat_mse.mean(axis=0)
feat_std_v  = val_feat_mse.std(axis=0) + 1e-8
ts_mean_v   = val_ts_mse.mean(axis=0)    # (144,): mean error at each window position
ts_std_v    = val_ts_mse.std(axis=0) + 1e-8

# Normalised eval scores
eval_feat_norm          = (feat_mse - feat_mean_v) / feat_std_v   # (n_wins, n_feat)
eval_ts_norm            = (ts_mse   - ts_mean_v)   / ts_std_v     # (n_wins, 144)
eval_scores_norm        = eval_feat_norm.mean(axis=1)
eval_scores_norm_max    = eval_feat_norm.max(axis=1)
# 95th-pct of position-normalised ts errors — sensitive to brief fault spikes
eval_scores_ts_norm_p95 = np.percentile(eval_ts_norm, 95, axis=1)

val_feat_norm   = (val_feat_mse - feat_mean_v) / feat_std_v
val_scores_norm = val_feat_norm.mean(axis=1)

# ── Window → timestep score conversion ─────────────────────────────────────
n_ts = len(y_true_timestep)

def wins_to_timestep(win_scores, n_timesteps, window_size=144):
    """Average the score of every window that overlaps each timestep."""
    out = np.zeros(n_timesteps)
    cnt = np.zeros(n_timesteps)
    for i, s in enumerate(win_scores):
        out[i:i + window_size] += s
        cnt[i:i + window_size] += 1
    return out / np.maximum(cnt, 1)

# ── Compare all scoring strategies ─────────────────────────────────────────
all_strategies = {
    "max-feat (raw)":  eval_scores_max_feat,
    "global-mean":     eval_scores_global,
    "max-ts":          eval_scores_max_ts,
    "p95-feat":        eval_scores_p95_feat,
    "norm-mean":       eval_scores_norm,
    "norm-max":        eval_scores_norm_max,
    "norm-ts-p95":     eval_scores_ts_norm_p95,  # position-normalised peak ts error
}

print(f"{'Strategy':22s}  {'ROC-AUC (win)':14s}  {'ROC-AUC (ts)':12s}  Note")
best_roc, best_name, best_win_scores = 0.0, "", None

for nm, sc in all_strategies.items():
    ts_sc     = wins_to_timestep(sc, n_ts)
    roc_win   = roc_auc_score(y_true_window, sc)
    roc_ts    = roc_auc_score(y_true_timestep, ts_sc)
    effective = max(roc_ts, 1 - roc_ts)
    flip      = roc_ts < 0.5
    note      = "↑FLIP" if flip else ""
    print(f"  {nm:22s}  {roc_win:.4f}          {roc_ts:.4f}  {note}")
    use_sc = -sc if flip else sc
    if effective > best_roc:
        best_roc, best_name, best_win_scores = effective, nm, use_sc

print(f"\nBest strategy: '{best_name}'  (effective ROC-AUC timestep = {best_roc:.4f})")

# ── FIX: convert best window scores → timestep scores ──────────────────────
best_ts_scores = wins_to_timestep(best_win_scores, n_ts)

# ── Adaptieve tune/eval split op mediaan-anomaliepositie ───────────────────
# Zorgt dat anomalieën in beide halves vallen voor eerlijke drempelkalibratie.
anom_positions_ts = np.where(y_true_timestep == 1)[0]
if len(anom_positions_ts) >= 4:
    split_ts = int(np.median(anom_positions_ts))
else:
    split_ts = n_ts // 2

y_tune_ts = y_true_timestep[:split_ts]
y_eval_ts = y_true_timestep[split_ts:]
print(f"Split: {split_ts}  | anomalies tune: {y_tune_ts.sum()}  eval: {y_eval_ts.sum()}")

# ── Temporele smoothing: driehoekskernel (5 punten) ────────────────────────
_kernel = np.array([1, 2, 3, 2, 1], dtype=float); _kernel /= _kernel.sum()
best_ts_scores = np.convolve(best_ts_scores, _kernel, mode="same")
sc_tune_ts = best_ts_scores[:split_ts]
sc_eval_ts = best_ts_scores[split_ts:]

# ── F2-maximalisatie op tune-set (recall weegt 2× zwaarder) ───────────────
from sklearn.metrics import fbeta_score as _fb
cands = np.percentile(sc_tune_ts, np.linspace(0.1, 99.9, 600))
best_f2_tune, best_thr    = 0.0, cands[-1]
best_f1_tune, best_thr_f1 = 0.0, cands[-1]
for t in cands:
    yp  = (sc_tune_ts > t).astype(int)
    f2v = _fb(y_tune_ts, yp, beta=2, zero_division=0)
    f1v = f1_score(y_tune_ts, yp, zero_division=0)
    if f2v > best_f2_tune: best_f2_tune, best_thr    = f2v, t
    if f1v > best_f1_tune: best_f1_tune, best_thr_f1 = f1v, t

print(f"F2-tune drempel (HVAC primair): {best_thr:.4f}  "
      f"(tune F2={best_f2_tune:.4f}, alarm rate={(sc_tune_ts > best_thr).mean():.3f})")
print(f"F1-tune drempel (referentie):   {best_thr_f1:.4f}  "
      f"(tune F1={best_f1_tune:.4f}, alarm rate={(sc_tune_ts > best_thr_f1).mean():.3f})")

# ── POT threshold op genormaliseerde val-scores (referentie) ────────────────
def pot_threshold(scores_clean, risk=0.01, tail_pct=0.10):
    s = np.sort(scores_clean)
    n_tail = max(int(len(s) * tail_pct), 20)
    exc = s[-n_tail:] - s[-n_tail]
    shape, _, scale = genpareto.fit(exc, floc=0)
    u = s[-n_tail]; q = n_tail / len(s)
    if abs(shape) > 1e-8:
        return float(u + (scale / shape) * ((risk / q) ** (-shape) - 1))
    return float(u - scale * np.log(risk / q))

pot_thr = pot_threshold(val_scores_norm)
print(f"POT threshold (normalised val): {pot_thr:.4f}")

### 6.4 Drempelkalibratie en Scorekaart

We vergelijken **Methode A** (beste scoring-strategie op ROC-AUC) met **Methode B** (norm-max z-score per feature + run-length filter) door de F2-score op de tune-set te vergelijken. De winnende methode wordt toegepast op de eval-set voor de finale scorekaart.

In [ ]:
def get_event_stats(y_true, y_pred):
    """Event-based metrics: recall, precision and F1 at the fault-event level."""
    def find_segs(y):
        segs, in_seg = [], False
        for i, v in enumerate(y):
            if v and not in_seg:
                in_seg, start = True, i
            elif not v and in_seg:
                segs.append((start, i - 1)); in_seg = False
        if in_seg: segs.append((start, len(y) - 1))
        return segs

    true_ev = find_segs(y_true)
    pred_ev = find_segs(y_pred)

    # How many true events are overlapped by at least one predicted event
    detected = sum(any(not (pe < ts or ps > te) for ps, pe in pred_ev) for ts, te in true_ev)
    # How many predicted events overlap at least one true event
    correct  = sum(any(not (te < ps or ts > pe) for ts, te in true_ev) for ps, pe in pred_ev)

    er = detected / len(true_ev) if true_ev else 0.0
    ep = correct  / len(pred_ev) if pred_ev else 0.0
    ef = 2 * ep * er / (ep + er) if (ep + er) > 0 else 0.0

    return {
        "total_events":    len(true_ev),
        "detected_events": detected,
        "n_pred_events":   len(pred_ev),
        "event_recall":    er,
        "event_precision": ep,
        "event_f1":        ef,
    }

In [ ]:
def build_reusable_scorecard(y_true, y_scores, threshold, model_name='Model', min_run=1):
    """Uniforme scorekaart -- zelfde 11 metrieken en volgorde in alle drie pipelines."""
    y_pred = (y_scores > threshold).astype(int)
    if min_run > 1:
        y_pred = run_length_filter(y_pred, min_run=min_run)

    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    f2      = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mcc     = matthews_corrcoef(y_true, y_pred)
    try:
        roc_auc = roc_auc_score(y_true, y_scores)
        pr_auc  = average_precision_score(y_true, y_scores)
    except Exception:
        roc_auc, pr_auc = 0, 0

    event_stats = get_event_stats(y_true, y_pred)

    metrics_data = [
        {'Metric': 'Precision',             'Value': p},
        {'Metric': 'Recall',                'Value': r},
        {'Metric': 'F1-Score',              'Value': f1},
        {'Metric': 'F2-Score (HVAC Focus)', 'Value': f2},
        {'Metric': 'Balanced Accuracy',     'Value': bal_acc},
        {'Metric': 'MCC',                   'Value': mcc},
        {'Metric': 'ROC-AUC',               'Value': roc_auc},
        {'Metric': 'PR-AUC',                'Value': pr_auc},
        {'Metric': 'Event Recall',          'Value': event_stats['event_recall']},
        {'Metric': 'Event Precision',       'Value': event_stats['event_precision']},
        {'Metric': 'Event F1',              'Value': event_stats['event_f1']},
    ]

    return {
        'metrics_df':       pd.DataFrame(metrics_data),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
        'y_pred_window':    y_pred,
        'event_details':    event_stats,
    }

In [ ]:
def run_length_filter(y_pred, min_run=3):
    """Suppress alarm runs shorter than min_run consecutive timesteps.

    Eliminates isolated single-spike false positives (sensor noise, brief
    transients) while preserving the sustained fault windows that real HVAC
    faults produce.
    """
    filtered = y_pred.copy().astype(int)
    i = 0
    while i < len(filtered):
        if filtered[i] == 1:
            run_start = i
            while i < len(filtered) and filtered[i] == 1:
                i += 1
            if (i - run_start) < min_run:
                filtered[run_start:i] = 0
        else:
            i += 1
    return filtered

In [ ]:
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                              balanced_accuracy_score, matthews_corrcoef,
                              roc_auc_score, average_precision_score, fbeta_score)

has_both = len(np.unique(y_eval_ts)) > 1

# ── Methode A: beste strategie uit ROC-AUC selectie ──────────────────────────
# F2 op tune-set is al berekend als best_f2_tune (in cel 6.3)

# ── Methode B: norm-max z-score per feature + run-length filter ───────────────
norm_max_ts      = wins_to_timestep(eval_scores_norm_max, n_ts)
norm_max_ts_smth = np.convolve(norm_max_ts, _kernel, mode="same")
sc_tune_nm = norm_max_ts_smth[:split_ts]
sc_eval_nm = norm_max_ts_smth[split_ts:]

cands_b = np.percentile(sc_tune_nm, np.linspace(0.1, 99.9, 600))
best_f2_b, best_thr_b = 0.0, cands_b[-1]
for t in cands_b:
    yp  = run_length_filter((sc_tune_nm > t).astype(int), min_run=MIN_RUN)
    f2v = _fb(y_tune_ts, yp, beta=2, zero_division=0)
    if f2v > best_f2_b:
        best_f2_b, best_thr_b = f2v, t

print(f"Methode A — {best_name:<30s}  tune F2 = {best_f2_tune:.4f}")
print(f"Methode B — norm-max z-score + RL({MIN_RUN})     tune F2 = {best_f2_b:.4f}")

# ── Kies beste methode op basis van tune-set F2 ───────────────────────────────
if best_f2_b >= best_f2_tune:
    print(f"→ Methode B gekozen (norm-max z-score + run-length filter)")
    final_sc_eval = sc_eval_nm
    final_thr     = best_thr_b
    final_ts_full = norm_max_ts_smth
    chosen_method = f"norm-max + RL({MIN_RUN})"
    y_pred_eval   = run_length_filter((sc_eval_nm > best_thr_b).astype(int), min_run=MIN_RUN)
else:
    print(f"→ Methode A gekozen ({best_name})")
    final_sc_eval = sc_eval_ts
    final_thr     = best_thr
    final_ts_full = best_ts_scores
    chosen_method = best_name
    y_pred_eval   = (sc_eval_ts > best_thr).astype(int)

# ── Scorekaart samenstellen ────────────────────────────────────────────────────
p, r, f1, _ = precision_recall_fscore_support(
    y_eval_ts, y_pred_eval, average="binary", zero_division=0)
f2  = fbeta_score(y_eval_ts, y_pred_eval, beta=2, zero_division=0)
ba  = balanced_accuracy_score(y_eval_ts, y_pred_eval)
mcc = matthews_corrcoef(y_eval_ts, y_pred_eval)
roc = roc_auc_score(y_eval_ts, final_sc_eval) if has_both else float("nan")
prc = average_precision_score(y_eval_ts, final_sc_eval) if has_both else float("nan")
evt = get_event_stats(y_eval_ts, y_pred_eval)

metrics_data = [
    ("Precision",             p),
    ("Recall",                r),
    ("F1-Score",              f1),
    ("F2-Score (HVAC Focus)", f2),
    ("Balanced Accuracy",     ba),
    ("MCC",                   mcc),
    ("ROC-AUC",               roc),
    ("PR-AUC",                prc),
    ("Event Recall",          evt["event_recall"]),
    ("Event Precision",       evt["event_precision"]),
    ("Event F1",              evt["event_f1"]),
]
tuned_result = {
    "metrics_df":       pd.DataFrame({"Metric": [m for m, v in metrics_data],
                                      "Value":  [v for m, v in metrics_data]}),
    "confusion_matrix": confusion_matrix(y_eval_ts, y_pred_eval),
    "y_pred_window":    y_pred_eval,
    "event_details":    evt,
    "threshold":        final_thr,
    "strategy":         chosen_method,
}
print(f"\nEvaluatie op tijdstap-niveau (eval-set)  |  methode: {chosen_method}  |  drempel: {final_thr:.4f}")
print(tuned_result["metrics_df"].to_string(index=False))
print(f"\nAlarm rate eval: {y_pred_eval.mean():.3f}  vs  GT: {y_eval_ts.mean():.3f}")
print(f"Event details: {evt}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
fig.suptitle(
    f"Encoder-Only Transformer — {chosen_method} | Tijdstap-niveau (eval-set)",
    fontsize=16, fontweight="bold"
)

# 1. Anomaliescore tijdserie (eval-set)
ax = axes[0, 0]
ax.plot(final_sc_eval, lw=1, color="#1f77b4", label="Anomaliescore")
ax.axhline(final_thr, color="#d62728", ls="--", lw=2, label=f"Drempel ({final_thr:.3f})")
anom_idx_eval = np.where(y_eval_ts == 1)[0]
ax.scatter(anom_idx_eval, final_sc_eval[anom_idx_eval], s=15, color="#ff7f0e",
           alpha=0.8, zorder=5, label="GT anomalie")
ax.set_title("Anomaliescore per tijdstap (eval-set)")
ax.set_xlabel("Tijdstap")
ax.legend(fontsize=10)
ax.grid(alpha=0.25)

# 2. Scoreverdeling normaal vs anomalie (eval-set)
ax = axes[0, 1]
for lbl, clr, nm in [(0, "#1f77b4", "Normaal"), (1, "#d62728", "Anomalie")]:
    mask = y_eval_ts == lbl
    if mask.any():
        sns.histplot(final_sc_eval[mask], bins=40, kde=True, stat="density",
                     color=clr, alpha=0.5, label=nm, ax=ax)
ax.axvline(final_thr, color="#2ca02c", ls="--", lw=2, label="Drempel")
ax.set_title("Scoreverdeling normaal vs. anomalie (eval-set)")
ax.legend(fontsize=10)
ax.grid(alpha=0.25)

# 3. Confusion matrix (eval-set)
ax = axes[1, 0]
sns.heatmap(tuned_result["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            cbar=False, square=True,
            xticklabels=["Pred norm", "Pred anomalie"],
            yticklabels=["Echt norm", "Echt anomalie"], ax=ax)
ax.set_title("Confusion matrix (eval-set)")

# 4. Metrics bar chart
ax = axes[1, 1]
mdf = tuned_result["metrics_df"].sort_values("Value", ascending=True)
sns.barplot(data=mdf, x="Value", y="Metric", hue="Metric",
            dodge=False, palette="viridis", legend=False, ax=ax)
for i, v in enumerate(mdf["Value"]):
    if not np.isnan(v):
        ax.text(min(v + 0.01, 1.05), i, f"{v:.3f}", va="center", fontsize=10)
ax.set_xlim(0, 1.15)
ax.set_title("Kernmetrieken (eval-set)")
plt.show()

### 6.5 Eindvisualisatie — Inferentie op volledige synthetische testset

Toont de anomaliescore en het binaire alarmsignaal (na run-length filter) over de volledige synthetische testset, inclusief de ground-truth annotaties. Identiek aan het afsluitende figuur in de DV-MPPCA notebook.

In [ ]:
anomalies_raw  = (final_ts_full > final_thr).astype(int)
anomalies_demo = run_length_filter(anomalies_raw, min_run=MIN_RUN)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, constrained_layout=True)

ax = axes[0]
ax.plot(final_ts_full, lw=1, color="#1f77b4", label=f"Anomaliescore ({chosen_method})")
ax.axhline(final_thr, color="#d62728", ls="--", lw=1.5,
           label=f"Drempel ({final_thr:.3f})")
ax.fill_between(range(len(final_ts_full)), final_ts_full.min(), final_thr,
                alpha=0.05, color="green", label="Normaalzone")
anom_idx = np.where(y_true_timestep == 1)[0]
ax.scatter(anom_idx, final_ts_full[anom_idx], s=12, color="#ff7f0e",
           alpha=0.7, label="GT anomalie")
ax.set_ylabel(chosen_method)
ax.set_title(f"Encoder-Only Transformer — Inferentie op synthetische testset ({GEBOUW})")
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

ax = axes[1]
ax.fill_between(range(len(anomalies_demo)), 0, anomalies_demo,
                color="#d62728", alpha=0.6, label=f"Voorspeld alarm (na RL filter min_run={MIN_RUN})")
ax.fill_between(range(len(y_true_timestep)), 0, [v * 0.5 for v in y_true_timestep],
                color="#ff7f0e", alpha=0.5, label="Ground truth")
ax.set_ylabel("Alarm (binair)")
ax.set_xlabel("Tijdstap")
ax.set_yticks([0, 1])
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.show()

n_pred = int(anomalies_demo.sum())
n_gt   = int(y_true_timestep.sum())
n_tot  = len(anomalies_demo)
print(f"Voorspeld alarm  : {n_pred}/{n_tot} ({n_pred/n_tot:.1%})")
print(f"Ground truth     : {n_gt}/{n_tot}  ({n_gt/n_tot:.1%})")